# C7-cnn-transfer — Practice p08 — Solution

The ten-child surface and stage interiors can be read without executing any module.

In [ ]:
# Cache pin (course convention, plan 009): pretrained weights live in the repo's
# gitignored reference/cache/ -- resolve it from the repo root BEFORE importing torch.
import os, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists())
os.environ["TORCH_HOME"] = str(_root / "reference" / "cache" / "torch")

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# float32 register (course exception): pretrained resnet50 is a float32 artifact.
# No float64 default here; inputs are cast .to(torch.float32) at the model
# boundary; repeat float32 forwards are bit-identical.
SEED = 20260804

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval()
assert next(model.parameters()).dtype == torch.float32

child_names = [name for name, _ in model.named_children()]
n_blocks = tuple(len(getattr(model, f"layer{i}")) for i in (1, 2, 3, 4))
mid_channels = tuple(getattr(model, f"layer{i}")[1].conv2.in_channels for i in (1, 2, 3, 4))
stride2_stages = [f"layer{i}" for i in (2, 3, 4)
                  if getattr(model, f"layer{i}")[0].conv2.stride == (2, 2)]
fc_in = int(model.fc.in_features)
fc_out = int(model.fc.out_features)


### Answer check

In [ ]:
assert child_names == ["conv1", "bn1", "relu", "maxpool", "layer1", "layer2", "layer3", "layer4", "avgpool", "fc"]
assert n_blocks == (3, 4, 6, 3)
assert mid_channels == (64, 128, 256, 512)
assert stride2_stages == ["layer2", "layer3", "layer4"]
assert (fc_in, fc_out) == (2048, 1000)
